In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
patients = spark.table("gold.dim_patient")
encounters = spark.table("gold.fact_encounter")
claims = spark.table("gold.fact_claims")
patient_conditions = spark.table("gold.fact_patient_conditions")

In [0]:
encounters = encounters.withColumn(
    "encounter_date_parsed",
    F.to_date(F.col("date_key").cast("string"), "yyyyMMdd")
)

In [0]:
w = Window.partitionBy("patient_key").orderBy("encounter_date_parsed")
encounter_labels = (
    encounters
    .select("patient_key", "encounter_date_parsed")
    .where(F.col("patient_key").isNotNull() & F.col("encounter_date_parsed").isNotNull())
    .withColumn("next_encounter_date", F.lead("encounter_date_parsed").over(w))
    .withColumn("days_to_next", F.datediff(F.col("next_encounter_date"), F.col("encounter_date_parsed")))
    .withColumn(
        "readmitted_30_days",
        F.when((F.col("days_to_next") > 0) & (F.col("days_to_next") <= 30), 1).otherwise(0)
    )
)

In [0]:
patient_labels = (
    encounter_labels
    .groupBy("patient_key")
    .agg(F.max("readmitted_30_days").alias("readmitted_30_days"))
)

In [0]:
encounter_features = (
    encounters
    .groupBy("patient_key")
    .agg(
        F.count("*").alias("encounter_count"),
        F.max("date_key").alias("last_encounter_date")
    )
)

claim_features = (
    claims
    .groupBy("patient_key")
    .agg(
        F.count("*").alias("claim_count"),
        F.sum("claim_amount").alias("total_claim_amount"),
        F.avg("claim_amount").alias("avg_claim_amount")
    )
)

condition_features = (
    patient_conditions
    .groupBy("patient_key")
    .agg(
        F.countDistinct("condition_key").alias("condition_count")
    )
)

patient_readmission_features = (
    patients.alias("p")
    .join(encounter_features.alias("e"), "patient_key", "left")
    .join(claim_features.alias("c"), "patient_key", "left")
    .join(condition_features.alias("d"), "patient_key", "left")
    .join(patient_labels, "patient_key", "left")
)

In [0]:
patient_readmission_features = patient_readmission_features.fillna({
    "encounter_count": 0,
    "claim_count": 0,
    "total_claim_amount": 0.0,
    "avg_claim_amount": 0.0,
    "condition_count": 0,
    "readmitted_30_days": 0
})

In [0]:
#display(patient_readmission_features.limit(1))

In [0]:
patient_readmission_features = (patient_readmission_features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.patient_readmission_features")
)